# 09 · Reducción de características V5 · Cinco modelos

Objetivo: evaluar cuántas de las **192 características V5** necesita cada uno de los cinco modelos.

Se prueban:

`K = 40, 60, 80, 100, 120, 150, 192`

para:

- Regresión Logística
- Random Forest
- XGBoost
- SVM RBF
- MLP

La selección de características se realiza **dentro de cada fold temporal** usando solo el
subentrenamiento del fold, mediante información mutua con el target.

Los hiperparámetros permanecen **congelados** con los mejores valores obtenidos en la
optimización intensiva V5.

> No se usa 2018–2021 para escoger K. Tampoco se toca 2022–2025 ni el holdout espacial.

## 0. Dependencias

In [1]:
%pip install -q scikit-learn xgboost joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [2]:
from pathlib import Path
import importlib
import json
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Imports listos.")

Imports listos.


## 2. Localizar proyecto

In [3]:
candidates = [
    Path.cwd(),
    Path.cwd() / "rain-threat-classifier",
    Path("/content/rain-threat-classifier"),
    Path("/content/drive/MyDrive/rain-threat-classifier"),
]

PROJECT_DIR = next(
    (
        p for p in candidates
        if (p / "resultados_completo" / "dataset_modelo_mensual_v3.csv").exists()
        and (p / "resultados_completo" / "columnas_modelo_v3.txt").exists()
        and (p / "resultados_completo" / "indicadores_mensuales_todas_zonas.csv").exists()
        and (p / "04_climatologia_era5land.py").exists()
        and (
            p
            / "resultados_experimentos"
            / "v5_optimizacion_intensiva"
            / "mejores_parametros_finales.json"
        ).exists()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró el proyecto o faltan archivos requeridos."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

clim = importlib.import_module("04_climatologia_era5land")
clim = importlib.reload(clim)

DATA_DIR = PROJECT_DIR / "resultados_completo"
TUNING_DIR = (
    PROJECT_DIR
    / "resultados_experimentos"
    / "v5_optimizacion_intensiva"
)
OUTPUT_DIR = (
    PROJECT_DIR
    / "resultados_experimentos"
    / "v5_reduccion_caracteristicas_todos"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_DIR: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier
OUTPUT_DIR: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier\resultados_experimentos\v5_reduccion_caracteristicas_todos


## 3. Cargar datos V3, climatología y parámetros V5

In [4]:
df = pd.read_csv(
    DATA_DIR / "dataset_modelo_mensual_v3.csv"
)
df["period_start"] = pd.to_datetime(df["period_start"])
df["target_period_start"] = pd.to_datetime(
    df["target_period_start"]
)

monthly = pd.read_csv(
    DATA_DIR / "indicadores_mensuales_todas_zonas.csv"
)
monthly["period_start"] = pd.to_datetime(
    monthly["period_start"]
)

features_v3 = [
    line.strip()
    for line in (
        DATA_DIR / "columnas_modelo_v3.txt"
    ).read_text(encoding="utf-8").splitlines()
    if line.strip()
]

features_v5 = (
    features_v3
    + clim.CLIMATE_FEATURE_NAMES
)

best_params = json.loads(
    (
        TUNING_DIR
        / "mejores_parametros_finales.json"
    ).read_text(encoding="utf-8")
)

assert len(features_v3) == 167
assert len(clim.CLIMATE_FEATURE_NAMES) == 25
assert len(features_v5) == 192
assert "target_amenaza" not in features_v5
assert "target_rx5day_mm" not in features_v5

print("Características V5:", len(features_v5))
print("Modelos con parámetros:", list(best_params))

Características V5: 192
Modelos con parámetros: ['Regresion_Logistica', 'Random_Forest', 'XGBoost', 'SVM_RBF', 'MLP']


## 4. Construir folds temporales con climatología sin leakage

In [5]:
train_df = df[
    df["split"] == "entrenamiento"
].copy()

label_encoder = LabelEncoder()
label_encoder.fit(
    train_df["target_amenaza"]
)
CLASS_NAMES = list(label_encoder.classes_)

TEMPORAL_FOLDS = [
    ("F1", "2004-12-01", "2005-01-01", "2007-12-01"),
    ("F2", "2007-12-01", "2008-01-01", "2010-12-01"),
    ("F3", "2010-12-01", "2011-01-01", "2013-12-01"),
    ("F4", "2013-12-01", "2014-01-01", "2017-12-01"),
]

fold_data = []

for name, train_end, val_start, val_end in TEMPORAL_FOLDS:
    subtrain = train_df[
        train_df["target_period_start"]
        <= pd.Timestamp(train_end)
    ].copy()

    subval = train_df[
        train_df["target_period_start"].between(
            pd.Timestamp(val_start),
            pd.Timestamp(val_end),
        )
    ].copy()

    reference = clim.fit_climatology(
        monthly=monthly,
        cutoff=subtrain["period_start"].max(),
        zones=subtrain["zone_id"].unique(),
    )

    X_train = clim.add_climate_features(
        frame=subtrain,
        base_X=subtrain[features_v3].copy(),
        reference=reference,
    )

    X_val = clim.add_climate_features(
        frame=subval,
        base_X=subval[features_v3].copy(),
        reference=reference,
    )

    y_train = label_encoder.transform(
        subtrain["target_amenaza"]
    )
    y_val = label_encoder.transform(
        subval["target_amenaza"]
    )

    assert X_train.shape[1] == 192
    assert X_val.shape[1] == 192

    fold_data.append({
        "name": name,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
    })

    print(
        name,
        "train=", X_train.shape,
        "val=", X_val.shape,
    )

F1 train= (1872, 192) val= (432, 192)
F2 train= (2304, 192) val= (432, 192)
F3 train= (2736, 192) val= (432, 192)
F4 train= (3168, 192) val= (576, 192)


## 5. Métricas

In [6]:
def calculate_metrics(y_true, y_pred):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=np.arange(len(CLASS_NAMES)),
            zero_division=0,
        )
    )

    result = {
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }

    for i, class_name in enumerate(CLASS_NAMES):
        key = class_name.lower()
        result[f"recall_{key}"] = float(recall[i])
        result[f"f1_{key}"] = float(f1[i])

    return result

## 6. Reconstruir los cinco modelos con hiperparámetros congelados

In [7]:
models = {
    "Regresion_Logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=5000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ]),

    "Random_Forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),

    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            probability=False,
            cache_size=2048,
            random_state=RANDOM_STATE,
        )),
    ]),

    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            solver="adam",
            max_iter=800,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}


def normalize_loaded_params(model_name, params):
    params = dict(params)

    if (
        model_name == "MLP"
        and "model__hidden_layer_sizes" in params
        and isinstance(
            params["model__hidden_layer_sizes"],
            list,
        )
    ):
        params["model__hidden_layer_sizes"] = tuple(
            params["model__hidden_layer_sizes"]
        )

    return params


for model_name in models:
    params = normalize_loaded_params(
        model_name,
        best_params[model_name],
    )
    models[model_name].set_params(**params)

    print("\n", model_name)
    print(params)


 Regresion_Logistica
{'model__class_weight': 'balanced', 'model__C': 0.001}

 Random_Forest
{'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.8, 'max_depth': 32, 'class_weight': 'balanced', 'bootstrap': True}

 XGBoost
{'subsample': 1.0, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'n_estimators': 450, 'min_child_weight': 2, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0.05, 'colsample_bytree': 0.85}

 SVM_RBF
{'model__gamma': 0.0005, 'model__class_weight': None, 'model__C': 3.0}

 MLP
{'model__validation_fraction': 0.15, 'model__n_iter_no_change': 20, 'model__learning_rate_init': 0.0001, 'model__hidden_layer_sizes': (192, 96, 48), 'model__batch_size': 128, 'model__alpha': 0.0003, 'model__activation': 'relu'}


## 7. Ranking de información mutua dentro de cada fold

Se calcula un ranking independiente por fold usando solo su `X_train` y `y_train`.

El mismo ranking model-agnostic se usa para comparar los cinco modelos con el mismo
subconjunto Top-K dentro de cada fold.

In [8]:
fold_rankings = {}

for fold in fold_data:
    mi = mutual_info_classif(
        fold["X_train"],
        fold["y_train"],
        random_state=RANDOM_STATE,
    )

    ranking = (
        pd.DataFrame({
            "feature": fold["X_train"].columns,
            "mutual_information": mi,
        })
        .sort_values(
            ["mutual_information", "feature"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    fold_rankings[fold["name"]] = ranking

    ranking.to_csv(
        OUTPUT_DIR
        / f"ranking_mi_{fold['name']}.csv",
        index=False,
    )

print("Rankings MI listos.")

Rankings MI listos.


## 8. Evaluar K para los cinco modelos

In [9]:
K_VALUES = [
    40,
    60,
    80,
    100,
    120,
    150,
    192,
]

rows = []
fold_rows = []

for model_name, base_model in models.items():
    print("\n" + "=" * 90)
    print(model_name)

    for k in K_VALUES:
        macro_scores = []
        balanced_scores = []
        alta_recalls = []
        media_f1 = []

        for fold in fold_data:
            if k == 192:
                selected = list(
                    fold["X_train"].columns
                )
            else:
                selected = (
                    fold_rankings[fold["name"]]
                    .head(k)["feature"]
                    .tolist()
                )

            model = clone(base_model)

            model.fit(
                fold["X_train"][selected],
                fold["y_train"],
            )

            pred = model.predict(
                fold["X_val"][selected]
            )

            metrics = calculate_metrics(
                fold["y_val"],
                pred,
            )

            macro_scores.append(
                metrics["macro_f1"]
            )
            balanced_scores.append(
                metrics["balanced_accuracy"]
            )
            alta_recalls.append(
                metrics["recall_alta"]
            )
            media_f1.append(
                metrics["f1_media"]
            )

            fold_rows.append({
                "modelo": model_name,
                "k_features": k,
                "fold": fold["name"],
                **metrics,
            })

        rows.append({
            "modelo": model_name,
            "k_features": k,
            "macro_f1_cv_mean": float(
                np.mean(macro_scores)
            ),
            "macro_f1_cv_std": float(
                np.std(macro_scores)
            ),
            "balanced_accuracy_cv_mean": float(
                np.mean(balanced_scores)
            ),
            "recall_alta_cv_mean": float(
                np.mean(alta_recalls)
            ),
            "f1_media_cv_mean": float(
                np.mean(media_f1)
            ),
        })

        print(
            f"K={k:3d} | "
            f"Macro F1={np.mean(macro_scores):.4f} "
            f"± {np.std(macro_scores):.4f} | "
            f"Recall Alta={np.mean(alta_recalls):.4f}"
        )

results = pd.DataFrame(rows)
fold_results = pd.DataFrame(fold_rows)

results.to_csv(
    OUTPUT_DIR / "comparacion_k_features_cv.csv",
    index=False,
)
fold_results.to_csv(
    OUTPUT_DIR / "resultados_por_fold_k.csv",
    index=False,
)

display(
    results.sort_values(
        ["modelo", "k_features"]
    )
)


Regresion_Logistica
K= 40 | Macro F1=0.3483 ± 0.0212 | Recall Alta=0.4020
K= 60 | Macro F1=0.3413 ± 0.0298 | Recall Alta=0.4003
K= 80 | Macro F1=0.3524 ± 0.0269 | Recall Alta=0.4009
K=100 | Macro F1=0.3522 ± 0.0233 | Recall Alta=0.4123
K=120 | Macro F1=0.3534 ± 0.0231 | Recall Alta=0.4222
K=150 | Macro F1=0.3520 ± 0.0105 | Recall Alta=0.4121
K=192 | Macro F1=0.3524 ± 0.0079 | Recall Alta=0.4105

Random_Forest
K= 40 | Macro F1=0.3471 ± 0.0291 | Recall Alta=0.3080
K= 60 | Macro F1=0.3452 ± 0.0290 | Recall Alta=0.2995
K= 80 | Macro F1=0.3462 ± 0.0342 | Recall Alta=0.3171
K=100 | Macro F1=0.3450 ± 0.0225 | Recall Alta=0.3218
K=120 | Macro F1=0.3401 ± 0.0248 | Recall Alta=0.3211
K=150 | Macro F1=0.3389 ± 0.0285 | Recall Alta=0.3015
K=192 | Macro F1=0.3484 ± 0.0241 | Recall Alta=0.3159

XGBoost
K= 40 | Macro F1=0.3393 ± 0.0115 | Recall Alta=0.3406
K= 60 | Macro F1=0.3487 ± 0.0277 | Recall Alta=0.3529
K= 80 | Macro F1=0.3515 ± 0.0201 | Recall Alta=0.3511
K=100 | Macro F1=0.3419 ± 0.0364 | Re

,modelo,k_features,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean,f1_media_cv_mean
28,MLP,40,0.353439,0.025433,0.357146,0.321105,0.312409
29,MLP,60,0.351455,0.014364,0.357813,0.378453,0.286691
30,MLP,80,0.344157,0.034056,0.346937,0.315128,0.317327
31,MLP,100,0.331427,0.006458,0.336970,0.344050,0.254048
32,MLP,120,0.343017,0.027032,0.350392,0.374182,0.305491
33,MLP,150,0.354613,0.030408,0.356260,0.360140,0.320518
34,MLP,192,0.364734,0.014346,0.367090,0.354574,0.343078
7,Random_Forest,40,0.347145,0.029090,0.350837,0.308026,0.347313
8,Random_Forest,60,0.345208,0.029032,0.348766,0.299484,0.340477
9,Random_Forest,80,0.346182,0.034165,0.348466,0.317138,0.343951


## 9. Mejor K por Macro F1 y regla de 1 error estándar

Para cada modelo se reportan dos opciones:

- `k_mejor_media`: K con mayor Macro F1 medio.
- `k_recomendado_1se`: menor K cuyo Macro F1 queda dentro de un error estándar
  del mejor resultado.

Esto permite preferir una versión más compacta cuando la pérdida de rendimiento es mínima.

In [10]:
selection_rows = []

for model_name in results["modelo"].unique():
    subset = results[
        results["modelo"] == model_name
    ].copy()

    best = subset.loc[
        subset["macro_f1_cv_mean"].idxmax()
    ]

    standard_error = (
        best["macro_f1_cv_std"]
        / np.sqrt(len(TEMPORAL_FOLDS))
    )

    threshold = (
        best["macro_f1_cv_mean"]
        - standard_error
    )

    eligible = (
        subset[
            subset["macro_f1_cv_mean"]
            >= threshold
        ]
        .sort_values("k_features")
    )

    recommended = eligible.iloc[0]

    selection_rows.append({
        "modelo": model_name,
        "k_mejor_media": int(
            best["k_features"]
        ),
        "mejor_macro_f1": float(
            best["macro_f1_cv_mean"]
        ),
        "mejor_recall_alta": float(
            best["recall_alta_cv_mean"]
        ),
        "error_estandar": float(
            standard_error
        ),
        "umbral_1se": float(
            threshold
        ),
        "k_recomendado_1se": int(
            recommended["k_features"]
        ),
        "macro_f1_k_1se": float(
            recommended["macro_f1_cv_mean"]
        ),
        "recall_alta_k_1se": float(
            recommended["recall_alta_cv_mean"]
        ),
    })

recommended_table = (
    pd.DataFrame(selection_rows)
    .sort_values(
        "mejor_macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(recommended_table)

recommended_table.to_csv(
    OUTPUT_DIR / "k_recomendado_por_modelo.csv",
    index=False,
)

,modelo,k_mejor_media,mejor_macro_f1,mejor_recall_alta,error_estandar,umbral_1se,k_recomendado_1se,macro_f1_k_1se,recall_alta_k_1se
0,MLP,192,0.364734,0.354574,0.007173,0.357562,192,0.364734,0.354574
1,SVM_RBF,100,0.361972,0.303526,0.012758,0.349213,40,0.360913,0.299873
2,XGBoost,192,0.359104,0.361089,0.014262,0.344842,60,0.348687,0.352865
3,Regresion_Logistica,120,0.353406,0.422172,0.011558,0.341848,40,0.348277,0.401986
4,Random_Forest,192,0.348398,0.315914,0.012052,0.336346,40,0.347145,0.308026


## 10. Ranking global de todas las combinaciones modelo + K

In [11]:
global_ranking = (
    results
    .sort_values(
        [
            "macro_f1_cv_mean",
            "recall_alta_cv_mean",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

display(global_ranking.head(20))

global_ranking.to_csv(
    OUTPUT_DIR
    / "ranking_global_modelo_k.csv",
    index=False,
)

,modelo,k_features,macro_f1_cv_mean,macro_f1_cv_std,balanced_accuracy_cv_mean,recall_alta_cv_mean,f1_media_cv_mean
0,MLP,192,0.364734,0.014346,0.367090,0.354574,0.343078
1,SVM_RBF,100,0.361972,0.025516,0.367083,0.303526,0.325041
2,SVM_RBF,40,0.360913,0.021033,0.367043,0.299873,0.325492
3,XGBoost,192,0.359104,0.028524,0.360921,0.361089,0.367192
4,SVM_RBF,80,0.358639,0.034066,0.363774,0.290028,0.331195
5,XGBoost,150,0.356090,0.025316,0.358174,0.340454,0.369232
6,SVM_RBF,192,0.355663,0.019487,0.358760,0.308811,0.333901
7,SVM_RBF,150,0.355548,0.026533,0.358327,0.306304,0.333811
8,SVM_RBF,120,0.355186,0.028923,0.360051,0.295497,0.327969
9,MLP,150,0.354613,0.030408,0.356260,0.360140,0.320518


## 11. Interpretación

Este notebook responde dos preguntas distintas:

1. **¿Qué modelo se beneficia más de reducir dimensionalidad?**
2. **¿Cuál es el menor número de características que conserva un rendimiento competitivo?**

Después de elegir el candidato `modelo + K` usando solo esta CV interna:

- se recalcula el ranking de información mutua con **todo 1991–2017**;
- se seleccionan las K mejores características;
- se entrena ese modelo con todo 1991–2017;
- se evalúa la versión reducida en 2018–2021;
- se compara contra V5 completa de 192 características.

Referencias actuales en validación 2018–2021:

- Regresión Logística V5: Macro F1 ≈ **0.4521**
- SVM V5: ≈ **0.4366**
- Random Forest V5: ≈ **0.4172**
- Persistencia: ≈ **0.4092**

Todavía no utilizar 2022–2025 ni holdout espacial.